<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_46_websocket_chat/note_lesson_46_chat_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 Урок 46 — WebSockets + практика: чат

Олена, Анна і Боб користуються `crispy_notes_project`. В Олени й Анни є група «Сім'я», і сьогодні в неї з'являється **чат**: повідомлення приходять одразу, без перезавантаження сторінки. Боб у групу не входить — і чат має це пам'ятати весь час, поки відкрито з'єднання.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | проєкт уроку 46 у ноутбуці: користувачі, група, «браузер» `WebsocketCommunicator` | 0.1 | |
| 1 | HTTP проти WebSocket; ASGI-застосунок вручну: `scope`, `receive`, `send`, відмова, `URLRouter` | 1.1–1.6 | |
| 2 | підключення до чату: учасник і чужий, `AuthMiddlewareStack` і анонім | 2.1–2.3 | |
| 3 | розсилка, кадри JSON, історія, ліміт і порядок, `database_sync_to_async` | 3.1–3.6 | |
| 4 | правила — у `services`, consumer лише перекладає їх на мову WebSocket | 4.1 | **Задача 1** |
| 5 | channel layer зсередини: канали, групи, `type` → метод, свій consumer, пам'ять чи Redis | 5.1–5.7 | **Задача 2** |
| 6 | права протягом з'єднання: події після COMMIT, вилучення, видалення групи, коди закриття, `Origin` | 6.1–6.8 | |
| 7 | XSS: хто екранує текст | 7.1–7.3 | |
| 8 | знайди помилку: перевірка лише в `connect()` | 8.1 | |
| 9 | практика: «хто зараз у чаті» | | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**: «браузери» Олени й Анни відкриваються на початку і живуть до кінця ноутбука, як вкладки.
- Клітинка з визначенням (`def`, `class`) лише створює функцію чи клас — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Сервер і браузер не потрібні: `WebsocketCommunicator` — це «браузер» з тестів, він говорить з consumer напряму, а channel layer живе в пам'яті процесу. Redis теж не потрібен.
- Зміни коду й діаграми — у книзі курсу: [Урок 46](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_46/); теорія — крок 7B Django-книги [Async Django і WebSocket](https://nikoriakviktor.github.io/notes_chat_app/tutorials/07_async_django/).

---
## 0. Проєкт у ноутбуці

У Colab клітинка завантажить папку уроку 46, встановить залежності (`channels`, `daphne`, `channels-redis` і все з уроку 45) і створить чисту базу SQLite. Redis не потрібен: без `REDIS_URL` проєкт бере `InMemoryChannelLayer`.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_46_websocket_chat"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import channels, crispy_bootstrap5, daphne, debug_toolbar, drf_spectacular, rest_framework_simplejwt  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt"], check=True)

os.chdir("crispy_notes_project")
for name in ("DATABASE_URL", "REDIS_URL"):
    os.environ.pop(name, None)                          # SQLite і channel layer у пам'яті
if os.path.exists("db.sqlite3"):
    os.remove("db.sqlite3")                             # щоразу — чиста база
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

Наступна клітинка налаштовує Django і створює героїв: Олену (створює групу «Сім'я»), Анну (учасниця) і Боба (не в групі). `DJANGO_ALLOW_ASYNC_UNSAFE` дозволяє писати звичайні запити ORM прямо в клітинках: Jupyter сам працює всередині циклу подій `asyncio` (урок 28). У приклад 3.6 ми цей дозвіл ненадовго приберемо й побачимо, від чого він захищає.

In [ ]:
import asyncio
import json

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в циклі подій asyncio (урок 28)
django.setup()

from channels.db import database_sync_to_async
from channels.layers import get_channel_layer
from channels.testing import WebsocketCommunicator
from django.conf import settings
from django.contrib.auth.models import User
from django.core.exceptions import PermissionDenied, ValidationError

from hello_app import selectors, services
from hello_app.consumers import CLOSE_NOT_ALLOWED, GroupChatConsumer
from hello_app.models import ChatMessage

olena = User.objects.create_user("olena", password="pass-12345")
ann = User.objects.create_user("ann", password="pass-12345")
bob = User.objects.create_user("bob", password="pass-12345")
family = services.create_group(name="Сім'я", creator=olena)
services.add_user_to_group(family, "ann")
print("channel layer:", settings.CHANNEL_LAYERS["default"]["BACKEND"])
print("група:", family.name, [u.username for u in family.user_set.order_by("username")])

### Приклад 0.1. «Браузер» у ноутбуці

`communicator_for(group, user)` відкриває сторінку чату групи від імені користувача — так само, як `tests_chat.py`. На сервері `scope["user"]` заповнює `AuthMiddlewareStack` (з cookie сесії), а `scope["url_route"]` — `URLRouter` (з адреси). Тут ми ставимо їх самі, тож consumer працює без сервера.

`drain(communicator)` дочитує всі кадри, які вже прийшли, і повертає їх списком: після кожного прикладу «вкладки» лишаються без непрочитаних повідомлень і не заважають наступним прикладам.

In [ ]:
def communicator_for(group, user, consumer=GroupChatConsumer):
    """«Браузер» користувача user на сторінці чату групи group — як у tests_chat.py."""
    communicator = WebsocketCommunicator(consumer.as_asgi(), f"/ws/groups/{group.pk}/chat/")
    communicator.scope["user"] = user                  # замість AuthMiddlewareStack
    communicator.scope["url_route"] = {"kwargs": {"group_pk": str(group.pk)}}  # замість URLRouter
    return communicator


async def drain(communicator, timeout=0.2):
    """Усі кадри, що вже прийшли (JSON → словники); порожній список — нічого не прийшло."""
    frames = []
    while not await communicator.receive_nothing(timeout=timeout):
        frames.append(await communicator.receive_json_from())
    return frames

In [ ]:
probe = communicator_for(family, olena)
print("шлях:", probe.scope["path"], "| user:", probe.scope["user"], "| url_route:", probe.scope["url_route"])

---
## 1. HTTP проти WebSocket

Чат на HTTP міг би **питати** сервер щосекунди: «є нові повідомлення?» (polling). WebSocket відкриває **одне** з'єднання, і сервер пише в нього сам, щойно є що сказати.

### Приклад 1.1. Скільки коштує «питати щосекунди»

Задача: повідомлення з'являється на сервері через 0,35 с. Порахуємо, скільки запитів зробить клієнт, що питає кожні 0,1 с і кожні 0,3 с, і на скільки він запізниться. Для порівняння — «push»: сервер сам кладе повідомлення в чергу, клієнт просто чекає (`asyncio.Queue`, урок 28).

In [ ]:
import time


async def polling(arrive_at, interval):
    """Клієнт питає кожні interval с; повідомлення з'являється на сервері в момент arrive_at.
    Повертає (скільки запитів, запізнення в секундах)."""
    start = time.perf_counter()
    requests = 0
    while True:
        requests += 1                                  # GET /messages?after=41
        waited = time.perf_counter() - start
        if waited >= arrive_at:
            return requests, waited - arrive_at
        await asyncio.sleep(interval)                  # «нового немає» — чекаємо до наступного запиту


async def push(arrive_at):
    """Одне з'єднання: сервер сам кладе повідомлення в чергу клієнта."""
    inbox = asyncio.Queue()
    start = time.perf_counter()

    async def server():
        await asyncio.sleep(arrive_at)
        await inbox.put("Привіт")

    task = asyncio.create_task(server())
    await inbox.get()
    await task
    return 1, time.perf_counter() - start - arrive_at

In [ ]:
for name, result in [("polling кожні 0,1 с", await polling(0.35, 0.1)),
                     ("polling кожні 0,3 с", await polling(0.35, 0.3)),
                     ("WebSocket (push)   ", await push(0.35))]:
    print(f"{name}: запитів/з'єднань {result[0]}, запізнення {result[1]:.2f} с")

Частий polling — багато порожніх запитів; рідкий — менше запитів, але повідомлення запізнюється майже на весь інтервал. Push дає одне з'єднання і майже нульове запізнення. Ціна — сервер тримає тисячі відкритих з'єднань одночасно. Потік на кожне (WSGI) цього не витримає, потрібен цикл подій — **ASGI**.

**Правило:** коли сервер має **сам** повідомляти клієнта (чат, сповіщення, онлайн-статус), беруть WebSocket, а не polling.

### Приклад 1.2. ASGI-застосунок вручну

ASGI-застосунок — це `async`-функція `app(scope, receive, send)`. Сервер (daphne, uvicorn) передає їй словник `scope` (хто і куди підключається), а далі всі події — теж словники: `receive()` дає наступну подію від браузера, `send(...)` відправляє подію браузеру. Ні Django, ні Channels тут ще немає.

Задача: найменший «відлунний» WebSocket — приймає з'єднання і повертає кожен текст великими літерами. `log` записує, які події отримав застосунок.

In [ ]:
log = []


async def echo_app(scope, receive, send):
    """WebSocket за протоколом ASGI: лише словники, без Django і Channels."""
    log.append(f"scope: {scope['type']}")
    while True:
        message = await receive()
        log.append(message["type"])
        if message["type"] == "websocket.connect":            # браузер просить з'єднання
            await send({"type": "websocket.accept"})            # …приймаємо
        elif message["type"] == "websocket.receive":            # браузер надіслав текст
            await send({"type": "websocket.send", "text": message["text"].upper()})
        elif message["type"] == "websocket.disconnect":         # браузер закрив вкладку
            return

In [ ]:
echo = WebsocketCommunicator(echo_app, "/ws/echo/")
print("connect:", await echo.connect())
await echo.send_to(text_data="привіт, сервере")
print("відповідь:", await echo.receive_from())
await echo.disconnect()
print("події застосунку:", log)

**🔮 Прогноз:** `connect()` повернув `(True, None)`. Що означає `None`?

<details>
<summary>Відповідь</summary>

Друге значення — обраний **підпротокол** (заголовок `Sec-WebSocket-Protocol`), якщо з'єднання прийняте, або **код закриття**, якщо відхилене. Наш застосунок підпротоколу не обирав — `None`. Події `websocket.connect` → `websocket.receive` → `websocket.disconnect` — це і є життєвий цикл, який `GroupChatConsumer` розкладає на методи `connect()`, `receive()`, `disconnect()`.
</details>

### Приклад 1.3. Що лежить у `scope`

`scope` — аналог `request` у view: шлях, заголовки рукостискання, адреса клієнта. Задача: застосунок, що зберігає `scope` і приймає з'єднання. Підключимося із заголовком `Origin` — адресою сторінки, яка відкриває з'єднання.

In [ ]:
seen = {}


async def scope_app(scope, receive, send):
    seen.update(scope)
    await receive()                                   # websocket.connect
    await send({"type": "websocket.accept"})
    await receive()                                   # websocket.disconnect

In [ ]:
spy = WebsocketCommunicator(scope_app, "/ws/groups/1/chat/", headers=[(b"origin", b"http://localhost:8000")])
await spy.connect()
await spy.disconnect()
for key in ("type", "path", "headers", "subprotocols"):
    print(f"{key:12} = {seen[key]!r}")
print("user у scope?", "user" in seen)

У сирому `scope` немає ні `user`, ні `url_route`. Їх додають «обгортки» навколо застосунку: `AuthMiddlewareStack` читає cookie з `headers` і кладе `scope["user"]`, `URLRouter` розбирає `path` і кладе `scope["url_route"]`. Саме тому `communicator_for` ставить їх вручну.

### Приклад 1.4. Відмова до `accept()`

Якщо застосунок відповідає на `websocket.connect` подією `websocket.close`, з'єднання так і не відкривається. Задача: фабрика застосунків, що завжди відмовляє з заданим кодом (або без коду).

In [ ]:
def rejecting_app(code=None):
    """ASGI-застосунок, який відмовляє в рукостисканні."""
    async def app(scope, receive, send):
        await receive()                                # websocket.connect
        await send({"type": "websocket.close"} if code is None else {"type": "websocket.close", "code": code})
    return app

In [ ]:
for code in (None, 4403):
    attempt = WebsocketCommunicator(rejecting_app(code), "/ws/x/")
    print(f"close(code={code}) до accept →", await attempt.connect())

Без коду `WebsocketCommunicator` повертає `1000` — звичайне закриття. Браузер у такому разі взагалі не бачить нашого коду: рукостискання не відбулося, і `onclose` отримує `1006`, як при обриві мережі. Код застосунку (`4403`) браузер побачить лише тоді, коли закриває **вже відкрите** з'єднання (приклад 6.4).

### Приклад 1.5. `URLRouter`: адреса → consumer і `url_route`

`hello_project/routing.py` описує WebSocket-адреси регулярним виразом: `^ws/groups/(?P<group_pk>\d+)/chat/$`. Задача: підставити замість `GroupChatConsumer` маленький consumer, який лише показує, що поклав `URLRouter` у `scope`.

In [ ]:
from channels.generic.websocket import AsyncWebsocketConsumer
from channels.routing import URLRouter
from django.urls import re_path

from hello_project.routing import websocket_urlpatterns


class RouteProbe(AsyncWebsocketConsumer):
    """Приймає з'єднання і надсилає, що знайшов URLRouter."""

    async def connect(self):
        await self.accept()
        await self.send(text_data=json.dumps(self.scope["url_route"]))


probe_router = URLRouter([re_path(websocket_urlpatterns[0].pattern.regex.pattern, RouteProbe.as_asgi())])
print("шаблон з routing.py:", websocket_urlpatterns[0].pattern.regex.pattern)

In [ ]:
probe = WebsocketCommunicator(probe_router, f"/ws/groups/{family.pk}/chat/")
await probe.connect()
print("url_route:", await probe.receive_json_from())
await probe.disconnect()

wrong = WebsocketCommunicator(URLRouter(websocket_urlpatterns), "/ws/groups/abc/chat/")
try:
    await wrong.connect()
except ValueError as error:
    print("адреса /ws/groups/abc/chat/ →", type(error).__name__ + ":", error)

**🔮 Прогноз:** `group_pk` у `url_route` — рядок `'1'`. Чому consumer пише `int(self.scope['url_route']['kwargs']['group_pk'])`?

<details>
<summary>Відповідь</summary>

Регулярний вираз повертає **текст** з адреси. `\d+` гарантує, що там лише цифри (тому `abc` не знайшов маршруту), але перетворити на число — справа consumer: далі `group_pk` порівнюється з `pk` моделі й іде в назву групи channel layer.
</details>

### Приклад 1.6. Шлях одного повідомлення

Тепер — справжній чат. Шлях повідомлення від Анни до Олени (consumer Анни і consumer Олени — два окремі об'єкти, їх з'єднує channel layer):

```text
connect()   Анна відкриває сторінку чату
   │  selectors.is_group_member(ann, family)?
   │      ├─ ні  → close() до accept()            → рукостискання не відбулось
   │      └─ так → group_add("chat_group_1", канал Анни) → accept() → кадри "history"
   ▼
receive()   Анна пише {"content": "Привіт!"}
   │  services.post_chat_message(...)              → ChatMessage у базі (або ValidationError / PermissionDenied)
   │  group_send("chat_group_1", {"type": "chat_message", ...})
   ▼
channel layer: група "chat_group_1"
   ├─► канал Олени → consumer Олени.chat_message() → кадр {"type": "message", ...} у браузер Олени
   └─► канал Анни  → consumer Анни.chat_message()  → той самий кадр у браузер Анни (своє ж)
```

---
## 2. Підключення до чату

### Приклад 2.1. Учасник і чужий

Задача: відкрити чат «Сім'ї» від Олени й від Боба. `await communicator.connect()` повертає пару `(підключено, код)`.

**🔮 Прогноз:** що отримає Боб — з'єднання, яке одразу закриється, чи відмову ще в рукостисканні?

In [ ]:
olena_ws = communicator_for(family, olena)            # «вкладка» Олени — живе до кінця ноутбука
bob_ws = communicator_for(family, bob)
olena_result = await olena_ws.connect()
bob_result = await bob_ws.connect()
print("Олена:", olena_result, "| Боб:", bob_result)

<details>
<summary>Відповідь</summary>

Боб отримує `(False, 1000)` — відмову в рукостисканні: consumer викликає `close()` **до** `accept()`, тож з'єднання так і не відкривається (як у прикладі 1.4). Перевірка — `selectors.is_group_member`, та сама, що для сторінок групи в уроці 45. Consumer не пише свого правила.
</details>

### Приклад 2.2. Хто такий `scope["user"]`

На сервері користувача визначає `AuthMiddlewareStack`: читає cookie `sessionid` із заголовків рукостискання і завантажує `User` з бази. Задача: consumer `WhoAmI`, загорнутий у `AuthMiddlewareStack`, відповідає, кого він «бачить», — з cookie Олени і без cookie.

In [ ]:
from channels.auth import AuthMiddlewareStack
from django.test import Client, override_settings

browser = Client()
browser.force_login(olena)                            # вхід на сайт → сесія в базі, cookie sessionid
olena_cookie = f"sessionid={browser.cookies['sessionid'].value}".encode()


class WhoAmI(AsyncWebsocketConsumer):
    async def connect(self):
        await self.accept()
        user = self.scope["user"]
        await self.send(text_data=json.dumps({"user": str(user), "authenticated": user.is_authenticated}))

In [ ]:
for label, headers in [("з cookie Олени", [(b"cookie", olena_cookie)]), ("без cookie", [])]:
    who = WebsocketCommunicator(AuthMiddlewareStack(WhoAmI.as_asgi()), "/ws/whoami/", headers=headers)
    await who.connect()
    print(f"{label:15} →", await who.receive_json_from())
    await who.disconnect()

Без cookie `scope["user"]` — не `None`, а `AnonymousUser`: у нього `is_authenticated == False`. Тому `is_group_member` починається з `user.is_authenticated and ...` — анонім не учасник жодної групи.

### Приклад 2.3. Повний стек з `asgi.py`

`websocket_application()` з `hello_project/asgi.py` збирає стек: `AllowedHostsOriginValidator` → `AuthMiddlewareStack` → `URLRouter`. Задача: підключитися через нього до чату «Сім'ї» з нашої сторінки (`Origin: http://localhost:8000`) — з cookie Олени і без cookie. `connect_with` знадобиться ще в розділі 6.

In [ ]:
from hello_project.asgi import websocket_application


async def connect_with(app, *, origin=b"http://localhost:8000", cookie=None, group=family):
    """Підключитися через ASGI-стек app як браузер: Host, Origin і (необов'язково) cookie сесії."""
    headers = [(b"host", b"localhost")]
    if origin is not None:
        headers.append((b"origin", origin))
    if cookie is not None:
        headers.append((b"cookie", cookie))
    communicator = WebsocketCommunicator(app, f"/ws/groups/{group.pk}/chat/", headers=headers)
    connected, _ = await communicator.connect()
    await communicator.disconnect()
    return connected

In [ ]:
with override_settings(ALLOWED_HOSTS=["localhost"]):
    stack = websocket_application()
print("з cookie Олени:", await connect_with(stack, cookie=olena_cookie))
print("анонім:        ", await connect_with(stack))

Той самий consumer, ті самі правила: анонім отримує відмову, бо `AuthMiddlewareStack` дав йому `AnonymousUser`, а `is_group_member` для нього `False`.

---
## 3. Розсилка й історія

### Приклад 3.1. Одне повідомлення — усій групі

Задача: Анна відкриває чат (`ann_ws` — її «вкладка» до кінця ноутбука). Олена надсилає `{"content": "  Привіт!  "}` (`send_json_to`). Читаємо кадр у Олени й у Анни (`receive_json_from`) і рахуємо повідомлення в базі.

In [ ]:
ann_ws = communicator_for(family, ann)
await ann_ws.connect()
await olena_ws.send_json_to({"content": "  Привіт!  "})
frame_olena = await olena_ws.receive_json_from()
frame_ann = await ann_ws.receive_json_from()
saved = await ChatMessage.objects.acount()
print("Олена:", frame_olena)
print("Анна: ", frame_ann)
print("у базі повідомлень:", saved, "| кадри однакові:", frame_olena == frame_ann)

**🔮 Прогноз:** куди поділися пробіли? Чому Олена отримала **своє** повідомлення?

<details>
<summary>Відповідь</summary>

Пробіли прибрав `services.post_chat_message` (`strip()`), тож правило тексту одне для будь-якого транспорту. Олена отримала своє повідомлення, бо `group_send` іде **всій** групі, разом з автором. Так браузер показує повідомлення лише тоді, коли сервер його прийняв і зберіг.
</details>

### Приклад 3.2. Кадр і рядок у базі

Кадр — це JSON, який JS-клієнт розбирає в `onmessage`. Задача: звірити кадр з рядком `ChatMessage`.

In [ ]:
row = await ChatMessage.objects.select_related("author").aget(pk=frame_ann["message_id"])
print("поля кадру:", sorted(frame_ann))
print("у базі:    ", row.author.username, repr(row.content), row.timestamp.isoformat())
print("timestamp у кадрі — рядок:", type(frame_ann["timestamp"]).__name__, "| збігається:",
      frame_ann["timestamp"] == row.timestamp.isoformat())

У кадрі лише JSON-сумісні значення: `message_id` (число), рядки і час у форматі ISO 8601. Об'єкт моделі чи `datetime` у WebSocket не передаси — тільки текст. Той самий формат потрібен і channel layer у Redis (приклад 5.6).

### Приклад 3.3. Ще два повідомлення

Анна дописує «Що купити?» і «Хліб». Кожне повідомлення отримують обидві вкладки — читаємо їх через `drain`, щоб нічого не лишилось непрочитаним.

In [ ]:
for text in ("Що купити?", "Хліб"):
    await ann_ws.send_json_to({"content": text})
print("Олена отримала:", [frame["content"] for frame in await drain(olena_ws)])
print("Анна отримала: ", [frame["content"] for frame in await drain(ann_ws)])

### Приклад 3.4. Історія в новій вкладці

Задача: Олена відкриває **другу вкладку** (`olena_tab2`). Одразу після `accept()` consumer надсилає кадри `history` — збираємо їхній `content` по порядку.

In [ ]:
olena_tab2 = communicator_for(family, olena)
await olena_tab2.connect()
history = [frame["content"] for frame in await drain(olena_tab2) if frame["type"] == "history"]
await olena_tab2.disconnect()
print("історія:", history)

<details>
<summary>Чому від старих до нових</summary>

`selectors.recent_chat_messages` бере 50 **найновіших** (`order_by('-timestamp', '-id')[:50]`) і перевертає їх — від старих до нових, як читають чат. Повертає **список**, а не QuerySet (чому — у прикладі 3.6).
</details>

### Приклад 3.5. Ліміт історії: які саме повідомлення

Задача: у групі «Книжковий клуб» п'ять повідомлень (пишемо через `services`, без WebSocket). Нова вкладка отримує історію з `limit=3`.

**🔮 Прогноз:** які три — перші три чи останні три? І в якому порядку?

In [ ]:
club = services.create_group(name="Книжковий клуб", creator=olena)
for number in range(1, 6):
    services.post_chat_message(group_pk=club.pk, author=olena, content=f"Повідомлення {number}")

In [ ]:
print("limit=3:", [row["content"] for row in selectors.recent_chat_messages(club.pk, limit=3)])
print("усі:    ", [row["content"] for row in selectors.recent_chat_messages(club.pk)])

<details>
<summary>Відповідь</summary>

Останні три, від старих до нових: `3, 4, 5`. Спершу сортування «найновіші першими» й зріз `[:3]` (SQL `ORDER BY timestamp DESC, id DESC LIMIT 3`), потім `reversed`. Якби зріз брали з сортування за зростанням, нова вкладка показувала б найдавніші повідомлення чату, а не останні. Другий ключ `-id` потрібен, бо два повідомлення можуть мати однаковий `timestamp`.
</details>

### Приклад 3.6. Чому список, а не QuerySet: `database_sync_to_async`

Consumer працює в циклі подій, а ORM Django — синхронний: запит до бази блокував би **всі** з'єднання процесу. Django це забороняє: у циклі подій ORM кидає `SynchronousOnlyOperation`. `database_sync_to_async(функція)` виконує функцію в окремому потоці.

QuerySet — **лінивий**: SQL виконується, коли його перебирають. Задача: на хвилину прибрати дозвіл `DJANGO_ALLOW_ASYNC_UNSAFE` і перебрати QuerySet у корутині — так, як consumer перебирав би `history`, якби selector повернув QuerySet.

In [ ]:
from django.core.exceptions import SynchronousOnlyOperation

lazy = ChatMessage.objects.filter(group=family).order_by("id").values_list("content", flat=True)  # SQL ще не було


async def read_in_loop(rows):
    """Як consumer: перебрати результат прямо в циклі подій."""
    return [content for content in rows]

In [ ]:
os.environ.pop("DJANGO_ALLOW_ASYNC_UNSAFE")
try:
    try:
        await read_in_loop(lazy)
    except SynchronousOnlyOperation as error:
        print("QuerySet у циклі подій →", type(error).__name__ + ":", str(error)[:70], "…")
    rows = await database_sync_to_async(list)(lazy)    # SQL виконано в потоці
    print("через database_sync_to_async(list) →", await read_in_loop(rows))
finally:
    os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

Тому `recent_chat_messages` повертає `list(...)`: SQL виконується **всередині** функції, яку consumer викликає через `database_sync_to_async`, а в цикл подій повертаються вже готові словники. Повернений QuerySet «вистрілив» би у `for msg in history` в `connect()` — і consumer падав би на кожному підключенні.

**Правило:** з асинхронного коду — лише `database_sync_to_async(функція)` (або async-методи ORM на кшталт `acount()`, `aget()`), і функція повертає готові дані, а не лінивий QuerySet.

---
## 4. Правила — у `services`

`services.post_chat_message(group_pk=…, author=…, content=…)` — одне правило для будь-якого транспорту: WebSocket, API, тест. Текст обрізається з країв; порожній або довший за 2000 символів — `ValidationError`; не учасник групи — `PermissionDenied`.

### Задача 1. Три відмови сервісу

Без WebSocket виклич `services.post_chat_message(group_pk=family.pk, author=…, content=…)` для кожної пари з `cases` і збери в `errors` назву класу винятку (`type(error).__name__`) або `None`, якщо повідомлення збереглося:

1. Олена, `"   "`;
2. Олена, `"я" * 2001`;
3. Боб, `"Привіт"`.

In [ ]:
cases = [(olena, "   "), (olena, "я" * 2001), (bob, "Привіт")]

# YOUR CODE HERE


In [ ]:
assert isinstance(errors, list) and len(errors) == 3, "errors — список з трьох значень, по одному на кожну пару з cases"
print(errors)
assert errors == ["ValidationError", "ValidationError", "PermissionDenied"], "порожнє і задовге — ValidationError, Боб — PermissionDenied"
assert not ChatMessage.objects.filter(author=bob).exists()
print("✅ Задача 1 розв'язана")

**🔮 Прогноз:** що зробить consumer з кожним із цих винятків?

<details>
<summary>Відповідь</summary>

`ValidationError` — проігнорує повідомлення. `PermissionDenied` — закриє з'єднання кодом `4403`: людина вже не учасник. Правило живе в сервісі, а consumer лише перекладає відповідь сервісу на мову WebSocket.
</details>

### Приклад 4.1. Погані кадри в чаті

Задача: Олена з вкладки надсилає чотири «погані» кадри — не JSON, JSON-список, словник без `content`, задовгий текст. Перевіримо, що нікому нічого не прийшло й у базі не з'явилось нових повідомлень.

In [ ]:
before = await ChatMessage.objects.acount()
await olena_ws.send_to(text_data="це не JSON")             # json.loads → JSONDecodeError
await olena_ws.send_to(text_data="[1, 2, 3]")              # список: .get → AttributeError
await olena_ws.send_json_to({"text": "інше поле"})          # content = '' → ValidationError
await olena_ws.send_json_to({"content": "я" * 2001})        # задовге → ValidationError
print("Олена отримала:", await drain(olena_ws), "| Анна отримала:", await drain(ann_ws))
print("нових повідомлень у базі:", await ChatMessage.objects.acount() - before)

Consumer нічого не відповів і не закрив з'єднання: зламаний кадр — не привід рвати чат. А ось `PermissionDenied` (розділ 6) — привід.

| Що сталося | Хто вирішує | Що робить consumer |
|---|---|---|
| кадр не JSON / не словник | consumer (це формат транспорту) | ігнорує |
| порожній або задовгий текст | `services` → `ValidationError` | ігнорує |
| автор не учасник | `services` → `PermissionDenied` | `close(4403)` |
| усе гаразд | `services` → `ChatMessage` | `group_send` усій групі |

---
## 5. Channel layer зсередини

`group_send` не знає про consumers. Він кладе **словник** у канал кожного учасника групи, а consumer, який слухає канал, викликає метод з ім'ям `type` (крапки → підкреслення). Без consumer'ів можна працювати з layer напряму: `new_channel()`, `group_add()`, `group_send()`, `receive()`.

### Задача 2. Подія через channel layer

Візьми `layer = get_channel_layer()`, створи канал (`await layer.new_channel()`) у змінній `channel`, додай його в групу `"demo"`, надішли в групу `{"type": "member.removed", "user_id": 42}` і прочитай подію з каналу (`await layer.receive(channel)`) у `event`. Збережи в `handler` ім'я методу, який її обробить (крапки → підкреслення). Наприкінці прибери канал з групи (`group_discard`).

In [ ]:
# YOUR CODE HERE


In [ ]:
assert isinstance(event, dict), "event — словник, який повернув await layer.receive(channel)"
print("подія:", event, "→ метод:", handler)
assert event == {"type": "member.removed", "user_id": 42}
assert handler == "member_removed" and hasattr(GroupChatConsumer, handler), "у GroupChatConsumer має бути такий метод"
print("✅ Задача 2 розв'язана")

Далі приклади працюють зі своїми каналами й групами — від Задачі 2 вони не залежать.

### Приклад 5.1. `group_discard`: відписався — не отримуєш

Задача: канал підписується на групу й одразу відписується; потім у групу йде подія. Чекаємо її на каналі 0,3 с.

**🔮 Прогноз:** подія прийде чи ні?

In [ ]:
layer = get_channel_layer()


async def receive_or_none(channel, timeout=0.3):
    """Подія з каналу або None, якщо за timeout секунд нічого не прийшло."""
    try:
        return await asyncio.wait_for(layer.receive(channel), timeout)
    except asyncio.TimeoutError:
        return None

In [ ]:
left = await layer.new_channel()
await layer.group_add("demo_discard", left)
await layer.group_discard("demo_discard", left)
await layer.group_send("demo_discard", {"type": "chat.message", "content": "хтось є?"})
print("ім'я каналу:", left, "| отримав:", await receive_or_none(left))

<details>
<summary>Відповідь</summary>

Не прийде: `group_send` доставляє лише тим каналам, що **зараз** у групі. Тому `GroupChatConsumer.disconnect()` обов'язково викликає `group_discard`: інакше layer продовжував би слати події в канал закритої вкладки (в пам'яті — до переповнення черги, у Redis — до закінчення терміну).
</details>

### Приклад 5.2. `type` → ім'я методу

Channels перетворює `type` події на ім'я методу функцією `get_handler_name`. Задача: перевірити кілька типів, зокрема некоректні.

In [ ]:
from channels.consumer import get_handler_name

for message in [{"type": "chat_message"}, {"type": "chat.message"}, {"type": "member.removed"},
                {"type": "_secret"}, {"user_id": 42}]:
    try:
        print(f"{str(message):26} → {get_handler_name(message)}")
    except ValueError as error:
        print(f"{str(message):26} → ValueError: {error}")

`chat.message` і `chat_message` ведуть до того самого методу. Ім'я з підкреслення на початку заборонене: інакше подія з layer могла б викликати «внутрішній» метод consumer'а.

### Приклад 5.3. Які імена груп дозволені

Задача: чому група чату — `chat_group_1`, а не назва групи нотаток?

In [ ]:
for name in (services.chat_group_name(family.pk), "чат Сім'ї", "chat group 1", "a" * 100):
    try:
        await layer.group_add(name, left)
        await layer.group_discard(name, left)
        print(f"{name[:20]!r:24} → можна")
    except TypeError as error:
        print(f"{name[:20]!r:24} → TypeError: {str(error)[:60]}…")

Лише ASCII-літери, цифри, `-`, `_`, `.` і довжина менше 100. Назва групи нотаток — кирилиця з пробілами й апострофом, і її можна перейменувати. `pk` не змінюється й завжди проходить — тому `services.chat_group_name(group_pk)`.

### Приклад 5.4. Свій consumer з обробником події

Задача: consumer «Термінові новини» підписується на групу `newsroom`, а метод `news_breaking` пересилає подію `news.breaking` у браузер. Подію надсилаємо **не з consumer**, а напряму через layer — так роблять `services` (приклад 6.3).

In [ ]:
class NewsFlash(AsyncWebsocketConsumer):
    async def connect(self):
        await self.channel_layer.group_add("newsroom", self.channel_name)
        await self.accept()

    async def disconnect(self, close_code):
        await self.channel_layer.group_discard("newsroom", self.channel_name)

    async def news_breaking(self, event):              # type: "news.breaking"
        await self.send(text_data=json.dumps({"title": event["title"]}, ensure_ascii=False))

In [ ]:
reader = WebsocketCommunicator(NewsFlash.as_asgi(), "/ws/news/")
await reader.connect()
await layer.group_send("newsroom", {"type": "news.breaking", "title": "Відкрито новий міст"})
print("браузер отримав:", await reader.receive_json_from())

### Приклад 5.5. Подія, для якої немає методу

**🔮 Прогноз:** що станеться, якщо в `newsroom` прийде `{"type": "weather.update"}`, а методу `weather_update` у `NewsFlash` немає?

In [ ]:
await layer.group_send("newsroom", {"type": "weather.update", "temp": 5})
try:
    await reader.receive_from()
except ValueError as error:
    print("consumer впав:", error)

<details>
<summary>Відповідь</summary>

Consumer **падає** з `ValueError: No handler for message type weather.update`, і з'єднання розривається: вкладка `reader` більше не працює. Кожен `type`, який шлють у групу, мусить мати обробник у **всіх** consumer'ах, підписаних на неї. Тому `GroupChatConsumer` має `member_removed` і `group_deleted`, навіть якщо вони здебільшого нічого не роблять.
</details>

### Приклад 5.6. У пам'яті працює, у Redis — ні

`InMemoryChannelLayer` копіює словник (`deepcopy`) і приймає будь-які значення. `RedisChannelLayer` перед відправкою пакує подію у `msgpack` — це майже JSON. Задача: та сама подія з `datetime` в обох випадках.

In [ ]:
from datetime import datetime

import msgpack

with_datetime = {"type": "chat.message", "timestamp": datetime(2026, 11, 17, 9, 30)}
box = await layer.new_channel()
await layer.send(box, with_datetime)
print("InMemory:", await layer.receive(box))
try:
    msgpack.packb(with_datetime)
except TypeError as error:
    print("msgpack (Redis):", type(error).__name__ + ":", error)
print("з isoformat():", msgpack.packb({**with_datetime, "timestamp": with_datetime["timestamp"].isoformat()})[:30], "…")

На ноутбуці з `InMemoryChannelLayer` подія з `datetime` пройшла б, а на сервері з Redis — ні. Тому consumer кладе в `group_send` `msg.timestamp.isoformat()`, а `services` — лише `user_id`, а не об'єкт `User`.

### Приклад 5.7. Пам'ять чи Redis: що обирає `settings.py`

`settings.py` бере `RedisChannelLayer`, якщо задано `REDIS_URL`, інакше — `InMemoryChannelLayer`. Задача: запустити Django двічі окремим процесом — без змінної і з нею — і подивитися `CHANNEL_LAYERS`. Redis для цього не потрібен: налаштування лише читаються, з'єднання не відкривається.

In [ ]:
SHOW_LAYERS = "import django; django.setup(); from django.conf import settings; print(settings.CHANNEL_LAYERS['default'])"


def channel_layers_with(redis_url=None):
    env = {key: value for key, value in os.environ.items() if key != "REDIS_URL"}
    env["DJANGO_SETTINGS_MODULE"] = "hello_project.settings"
    if redis_url:
        env["REDIS_URL"] = redis_url
    return subprocess.run([sys.executable, "-c", SHOW_LAYERS], capture_output=True, text=True, env=env).stdout.strip()

In [ ]:
print("без REDIS_URL:", channel_layers_with())
print("з REDIS_URL:  ", channel_layers_with("redis://localhost:6379/0"))

| | `InMemoryChannelLayer` | `RedisChannelLayer` |
|---|---|---|
| Де живуть групи | у пам'яті процесу | у Redis |
| Кілька процесів сервера | ні: Олена на процесі 1 не почує Анну на процесі 2 | так |
| Коли | розробка, тести, ноутбук | сервер (кілька процесів у Docker) |

### Приклад 5.8. Групи не змішуються

Задача: Олена створює групу «Робота» з Бобом, Боб відкриває її чат. Олена пише в «Сім'ю». Перевіряємо: Анна отримала повідомлення, а Боб — нічого.

In [ ]:
work = services.create_group(name="Робота", creator=olena)
services.add_user_to_group(work, "bob")
bob_work = communicator_for(work, bob)
await bob_work.connect()
await olena_ws.send_json_to({"content": "Тільки для сім'ї"})
await drain(olena_ws)
ann_got = [frame["content"] for frame in await drain(ann_ws)]
bob_silent = await bob_work.receive_nothing(timeout=0.3)
await bob_work.disconnect()
print("Анна:", ann_got, "| Боб нічого не отримав:", bob_silent)
print("групи channel layer:", services.chat_group_name(family.pk), "і", services.chat_group_name(work.pk))

---
## 6. Права — протягом усього з'єднання

WebSocket живе годинами. Перевірка в `connect()` — це лише «на вході». Коли права змінюються (учасника вилучили, групу видалили), `services` надсилає подію в channel layer, і відкриті consumers реагують.

```text
services.remove_user_from_group(family, ann)
   ├─ group.user_set.remove(ann)                     ← зміна в базі
   └─ transaction.on_commit(                         ← подія — лише після COMMIT
          group_send("chat_group_1", {"type": "member.removed", "user_id": ann.pk}))
                 ├─► consumer Олени.member_removed() → не мій user_id → нічого
                 └─► consumer Анни.member_removed()  → мій!          → close(4403)
```

### Приклад 6.1. Чому сервіс викликають через `database_sync_to_async`

`remove_user_from_group` — синхронний і всередині викликає `async_to_sync(layer.group_send)`. Задача: виклик **прямо в клітинці** (а клітинка вже працює в циклі подій) — для нового учасника Дмитра.

In [ ]:
dan = User.objects.create_user("dan", password="pass-12345")
services.add_user_to_group(family, "dan")
try:
    services.remove_user_from_group(family, dan)
except RuntimeError as error:
    print("RuntimeError:", str(error)[:75], "…")
print("Дмитро в групі:", selectors.is_group_member(dan, family.pk))

База вже змінилась, а подія не пішла: `async_to_sync` не можна викликати в потоці, де працює цикл подій. З consumer чи асинхронного коду синхронний сервіс викликають через `await database_sync_to_async(сервіс)(...)`: він працює в окремому потоці, і там `async_to_sync` передає `group_send` назад у цикл подій.

### Приклад 6.2. Відкат транзакції — події немає

`notify_chat` надсилає подію через `transaction.on_commit`. Задача: «шпигунський» канал слухає групу «Сім'ї»; Дмитра повертаємо в групу, а тоді вилучаємо всередині `transaction.atomic()`, яка падає.

**🔮 Прогноз:** чи отримає шпигун `member.removed`? Чи лишиться Дмитро в групі?

In [ ]:
from django.db import transaction


def remove_then_fail(group, user):
    """Вилучення в транзакції, яка потім падає → ROLLBACK."""
    with transaction.atomic():
        services.remove_user_from_group(group, user)
        raise RuntimeError("помилка після вилучення")

In [ ]:
services.add_user_to_group(family, "dan")
spy_channel = await layer.new_channel()
await layer.group_add(services.chat_group_name(family.pk), spy_channel)
try:
    await database_sync_to_async(remove_then_fail)(family, dan)
except RuntimeError as error:
    print("транзакція впала:", error)
print("шпигун отримав:", await receive_or_none(spy_channel), "| Дмитро в групі:", selectors.is_group_member(dan, family.pk))

<details>
<summary>Відповідь</summary>

Події немає, і Дмитро досі в групі: транзакцію відкотили, а колбек `on_commit` при відкаті **відкидається**. Якби подію слали одразу (до COMMIT), consumer Дмитра закрився б, хоча в базі він лишився учасником, або, навпаки, consumer перевірив би членство ще до того, як зміну видно іншим з'єднанням.
</details>

### Приклад 6.3. Після COMMIT — подія всім consumers групи

Задача: та сама дія без помилки. Шпигун отримує подію; вкладки Олени й Анни — теж (їхні consumers у тій самій групі), але Дмитро — не вони, тож ніхто не закривається і в браузери нічого не йде.

In [ ]:
await database_sync_to_async(services.remove_user_from_group)(family, dan)
print("шпигун отримав:", await receive_or_none(spy_channel))
print("Олені в браузер:", await drain(olena_ws), "| Анні:", await drain(ann_ws))
await layer.group_discard(services.chat_group_name(family.pk), spy_channel)

### Приклад 6.4. Анну вилучили — її вкладка закривається

Задача: Олена вилучає Анну з групи. Що отримала вкладка Анни (`receive_output` — «сира» подія ASGI), і що Олена?

In [ ]:
await database_sync_to_async(services.remove_user_from_group)(family, ann)
ann_output = await ann_ws.receive_output(timeout=1)
olena_silent = await olena_ws.receive_nothing(timeout=0.3)
print("Анна:", ann_output, "| Олені нічого:", olena_silent)

<details>
<summary>Чому закрилась лише Анна</summary>

`remove_user_from_group` після COMMIT надсилає в групу подію `member.removed` з `user_id` Анни. Її отримують **усі** consumers групи, але закривається лише той, чий `self.user.pk` збігається. Код `4403` — з діапазону застосунку (4000–4999). З'єднання вже було відкрите, тож браузер побачить саме цей код у `onclose`.
</details>

### Приклад 6.5. Групу видалили — чат закривається

Задача: Олена створює тимчасову групу, відкриває її чат і видаляє групу (`services.delete_group`).

In [ ]:
temp = services.create_group(name="Тимчасова", creator=olena)
temp_ws = communicator_for(temp, olena)
await temp_ws.connect()
await database_sync_to_async(services.delete_group)(temp)
print("вкладка тимчасової групи:", await temp_ws.receive_output(timeout=1))
await temp_ws.disconnect()

`group.deleted` обробляє `GroupChatConsumer.group_deleted` — без перевірки `user_id`: чату більше немає ні для кого.

### Приклад 6.6. Коди закриття: перепідключатись чи ні

JS-клієнт (`static/hello_app/js/group_chat.js`) у `onclose` вирішує, що робити далі. Задача: та сама логіка на Python — для кодів, які ми вже бачили.

In [ ]:
MAX_FAILED_ATTEMPTS = 5


def after_close(code, failed_attempts=0):
    """Що зробить group_chat.js після onclose(code)."""
    if 4000 <= code < 5000:
        return "forbidden: не перепідключаємось (перепідключення поверне ту саму відмову)"
    if code in (1000, 1001):
        return "закрито навмисно: не перепідключаємось"
    if failed_attempts + 1 > MAX_FAILED_ATTEMPTS:
        return "gave_up: досить спроб"
    delay = min(1 * 2 ** failed_attempts, 30)          # 1 → 2 → 4 → … → 30 с (+ випадкові 0–1 с)
    return f"перепідключення приблизно через {delay} с"

In [ ]:
for code, attempts in [(1000, 0), (1001, 0), (1006, 0), (1006, 3), (1006, 5), (1011, 1), (CLOSE_NOT_ALLOWED, 0)]:
    print(f"{code} (невдалих спроб: {attempts}) → {after_close(code, attempts)}")

**🔮 Прогноз:** чому відмова в рукостисканні (приклад 2.1) і `4403` мають різні наслідки для клієнта?

<details>
<summary>Відповідь</summary>

Відмову в рукостисканні браузер бачить як `1006` — так само, як обрив мережі, тож клієнт пробує ще кілька разів і зупиняється на `gave_up`. `4403` приходить у вже відкрите з'єднання — клієнт точно знає, що доступу немає, і одразу показує «Немає доступу до чату цієї групи».
</details>

### Приклад 6.7. `Origin`: звідки прийшло з'єднання

Cookie сесії браузер надсилає на наш сайт з **будь-якої** сторінки. Без перевірки `Origin` чужий сайт відкрив би чат від імені Олени й читав би історію (Cross-Site WebSocket Hijacking). Задача: з cookie Олени підключитися зі своєї сторінки, з `https://evil.example` і без заголовка `Origin`.

In [ ]:
with override_settings(ALLOWED_HOSTS=["localhost"]):
    stack = websocket_application()
own = await connect_with(stack, origin=b"http://localhost:8000", cookie=olena_cookie)
evil = await connect_with(stack, origin=b"https://evil.example", cookie=olena_cookie)
no_origin = await connect_with(stack, origin=None, cookie=olena_cookie)
print("своя сторінка:", own, "| чужий сайт:", evil, "| без Origin:", no_origin)

Cookie однакова в усіх трьох спробах — різниця лише в `Origin`. `AllowedHostsOriginValidator` стоїть **першим** у стеку й відмовляє ще до того, як `AuthMiddlewareStack` завантажить користувача.

### Приклад 6.8. Валідатор читає налаштування один раз

**🔮 Прогноз:** чому `websocket_application()` — функція, а не просто змінна з готовим стеком? Задача: стек, зібраний під `ALLOWED_HOSTS=["chat.example"]`, і стек, зібраний після цього, — з'єднання з `https://chat.example`.

In [ ]:
with override_settings(ALLOWED_HOSTS=["chat.example"]):
    frozen = websocket_application()                   # налаштування прочитано тут
fresh = websocket_application()                        # DEBUG=True і ALLOWED_HOSTS=[] → лише localhost
print("зібраний під chat.example:", await connect_with(frozen, origin=b"https://chat.example", cookie=olena_cookie))
print("зібраний після:           ", await connect_with(fresh, origin=b"https://chat.example", cookie=olena_cookie))

<details>
<summary>Відповідь</summary>

`AllowedHostsOriginValidator` читає `ALLOWED_HOSTS` (і `DEBUG`) **один раз**, коли його створюють. Стек, зібраний при імпорті `asgi.py`, не побачив би `override_settings`. Тому тести збирають стек заново через `websocket_application()`. На сервері це означає: без `DJANGO_ALLOWED_HOSTS` чат не підключиться з жодної сторінки.
</details>

---
## 7. XSS: хто екранує

### Приклад 7.1. Сервер повертає текст як є

Задача: Олена надсилає в чат HTML з обробником `onerror`. Що поверне сервер і що збереже в базі?

In [ ]:
payload = '<img src=x onerror="alert(\'x\')">&'
await olena_ws.send_json_to({"content": payload})
echoed = (await olena_ws.receive_json_from())["content"]
stored = (await ChatMessage.objects.order_by("-id").afirst()).content
print("сервер повернув:", echoed)
print("у базі те саме:", stored == payload == echoed)

Сервер зберігає текст **як є** — він не знає, де його покажуть (HTML, JSON API, push-сповіщення). Екранує той, хто вставляє в HTML: JS-клієнт перед `innerHTML` викликає `escapeHtml`.

### Приклад 7.2. `escapeHtml` з `group_chat.js` на Python

Та сама заміна, що в JS: `&`, `<`, `>`, `"`, `'` → `&amp;`, `&lt;`, `&gt;`, `&quot;`, `&#039;`, **саме в такому порядку**.

In [ ]:
import html


def escape_html(text):
    """Як escapeHtml у group_chat.js: & — першим."""
    return (str(text).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
            .replace('"', "&quot;").replace("'", "&#039;"))


def escape_html_wrong(text):
    """Та сама заміна, але & — останнім."""
    return (str(text).replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")
            .replace("'", "&#039;").replace("&", "&amp;"))

In [ ]:
print("правильно:", escape_html(echoed))
print("& останнім:", escape_html_wrong(echoed))
print("браузер покаже «<» як:", html.unescape(escape_html_wrong("<")), "| правильно:", html.unescape(escape_html("<")))

**🔮 Прогноз:** чому `&` замінюють **першим**?

<details>
<summary>Відповідь</summary>

Інакше `<` → `&lt;`, а потім `&` у ньому → `&amp;lt;`: у браузері замість `<` з'явиться текст `&lt;`. Атаки це не відкриває, але ламає кожне повідомлення з `<`, `>` чи лапками. Тест `JsEscapeTests` у проєкті запускає справжню функцію з `group_chat.js` у Node.js.
</details>

### Приклад 7.3. Готові функції: `html.escape` і Django

Задача: порівняти нашу функцію з `html.escape` зі стандартної бібліотеки і `django.utils.html.escape` (її використовують шаблони з автоекрануванням).

In [ ]:
from django.utils.html import escape as django_escape

for name, function in [("escape_html", escape_html), ("html.escape", html.escape), ("django escape", django_escape)]:
    print(f"{name:14} → {function(payload)}")

Різниця лише в записі апострофа (`&#039;` чи `&#x27;`) — браузер показує обидва як `'`. У шаблонах Django екранує сам (`{{ message.content }}`); у JS, що будує HTML рядком для `innerHTML`, — лише ми.

---
## 8. Знайди помилку

### Приклад 8.1. Перевірка лише в `connect()`

`OldConsumer` — consumer зі стартового коду чату: членство перевіряється лише в `connect()`, повідомлення зберігаються напряму, подію вилучення він не обробляє. Його тест «чужого не пускає» зелений.

Задача: Анну повертаємо в групу і підключаємо через `OldConsumer`. Олена вилучає Анну, а Анна пише «Ще тут?». Чи лишилась Анна в групі, і чи збереглося її повідомлення?

In [ ]:
class OldConsumer(GroupChatConsumer):
    async def receive(self, text_data):
        content = json.loads(text_data).get("content", "").strip()
        if not content or len(content) > 2000:
            return
        msg = await database_sync_to_async(ChatMessage.objects.create)(
            group_id=self.group_pk, author=self.user, content=content)
        await self.channel_layer.group_send(self.room_group_name, {
            "type": "chat_message", "message_id": msg.id, "author": self.user.username,
            "content": msg.content, "timestamp": msg.timestamp.isoformat()})

    async def member_removed(self, event):
        pass                                           # стартовий код такої події не знав

In [ ]:
services.add_user_to_group(family, "ann")
old_ann = communicator_for(family, ann, consumer=OldConsumer)
await old_ann.connect()
await drain(old_ann)                                   # історія
await database_sync_to_async(services.remove_user_from_group)(family, ann)
await old_ann.send_json_to({"content": "Ще тут?"})
olena_saw = [frame["content"] for frame in await drain(olena_ws)]
await drain(old_ann)
still_member = selectors.is_group_member(ann, family.pk)
saved_after_removal = await ChatMessage.objects.filter(author=ann, content="Ще тут?").acount()
await old_ann.disconnect()
print("Анна в групі:", still_member, "| Олена побачила:", olena_saw, "| збережено від Анни:", saved_after_removal)

<details>
<summary>Відповідь</summary>

Членство перевірено **один раз**. Хто був учасником при підключенні, той лишається ним для цього consumer: його канал досі в групі channel layer (отримує нові повідомлення), а `receive` не перевіряє нічого. Тест дивився лише на **момент підключення**. У HTTP кожен запит — новий «connect», тож цього досить. У WebSocket права перевіряють на кожне повідомлення (`services.post_chat_message` → `PermissionDenied` → `close(4403)`) і за подією зміни прав (`member.removed`), як у прикладі 6.4.
</details>

---
## 9. Практика: «хто зараз у чаті»

### Задача 3. Presence

Коли учасник підключається чи відключається, усі в групі отримують кадр `{"type": "presence", "user": "<ім'я>", "online": true/false}`. Це подія **транспорту**: у базу нічого не пишемо, право доступу вже перевірено в `connect()` батьківського класу.

Напиши `PresenceConsumer(GroupChatConsumer)`:

- `connect()` — викликає `super().connect()`, і якщо з'єднання прийняте (з'явився `self.room_group_name`), розсилає групі подію `presence` з `online: True`;
- `disconnect(close_code)` — якщо `room_group_name` є, розсилає `online: False`, потім викликає `super().disconnect(close_code)`;
- `presence(event)` — пересилає подію в браузер як JSON.

Перевірка — група «Похід» з Оленою й Анною: Олена отримує свою presence, потім presence Анни (`online: true` і `false`), а Боб (не учасник) не підключається і нікому нічого не розсилає.

In [ ]:
hike = services.create_group(name="Похід", creator=olena)
services.add_user_to_group(hike, "ann")
print("група:", hike.name, [u.username for u in hike.user_set.order_by("username")])

In [ ]:
# YOUR CODE HERE


In [ ]:
assert issubclass(PresenceConsumer, GroupChatConsumer), "PresenceConsumer має успадковувати GroupChatConsumer"
olena_hike = communicator_for(hike, olena, consumer=PresenceConsumer)
ann_hike = communicator_for(hike, ann, consumer=PresenceConsumer)
bob_hike = communicator_for(hike, bob, consumer=PresenceConsumer)
try:
    assert (await olena_hike.connect())[0], "Олена — учасниця «Походу», її мають підключити"
    own = await olena_hike.receive_json_from()
    await ann_hike.connect()
    came = await olena_hike.receive_json_from()
    assert (await bob_hike.connect())[0] is False, "Боб не в групі — відмова в рукостисканні"
    await ann_hike.disconnect()
    left_frame = await olena_hike.receive_json_from()
    print("своя:", own, "\nприйшла:", came, "\nпішла:", left_frame)
    assert own == {"type": "presence", "user": "olena", "online": True}
    assert came == {"type": "presence", "user": "ann", "online": True}
    assert left_frame == {"type": "presence", "user": "ann", "online": False}
    assert await olena_hike.receive_nothing(timeout=0.3), "від Боба нічого не має прийти"
finally:
    await olena_hike.disconnect()
print("✅ Задача 3 розв'язана")

Олена отримує й **власну** presence: `connect` розсилає всій групі, куди consumer уже доданий. Це зміна **протоколу**: клієнти й тести, які чекали першим кадром `message` або тишу, тепер отримують `presence`. JS-клієнт кадри невідомого типу пропускає, тож щоб показати «онлайн», йому потрібна окрема гілка в `onmessage` (розбір — у книзі, розділ «Практика»).

### Прибирання

Закриваємо вкладки, які жили весь ноутбук.

In [ ]:
for communicator in (olena_ws, ann_ws):
    await communicator.disconnect()
print("повідомлень у чаті «Сім'ї»:", len(selectors.recent_chat_messages(family.pk)))

---
## Самоперевірка

1. Чим WebSocket відрізняється від HTTP-запиту і чому для чату не досить опитувати сервер?
2. Що лежить у сирому `scope` і хто додає `scope["user"]` та `scope["url_route"]`?
3. Навіщо `database_sync_to_async` і чому selector повертає список, а не QuerySet?
4. Що кладе `group_send` у канал і як consumer вирішує, який метод викликати? Що буде, якщо методу немає?
5. Чому `services` надсилає подію через `transaction.on_commit`?
6. Назви всі місця, де перевіряються права в чаті, від рукостискання до вилучення з групи.
7. Коли `InMemoryChannelLayer` перестає працювати і що тоді дає Redis?

## Далі

- Книга курсу: [Урок 46](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_46/) — ASGI-стек, покрокові діаграми з'єднання й вилучення, архітектура до/після.
- Наживо: у папці `crispy_notes_project` — `python manage.py runserver` (daphne), дві вкладки різних користувачів на сторінці групи → «Чат». З Redis: `docker compose up -d redis` і `REDIS_URL=redis://localhost:6379/0`.
- Тести: `python manage.py test hello_app.tests_chat hello_app.tests_consumers`.